In [ ]:
import mediapipe as mp
import os
import mmap
import struct
import numpy as np
import json
import time

FRAME_PATH = "/dev/shm/latest_rgb.npy"

MODEL_PATH = "/home/xilinx/jupyter_notebooks/FASTAR/_DEMO/gesture_recognizer.task"

BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

print("Booting MediaPipe and loading weights into RAM...")

options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO, 
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

with GestureRecognizer.create_from_options(options) as recognizer:
    print("Model ready! Waiting for frames...")
    start_time = int(time.perf_counter() * 1000)
    

    while True:
        if not os.path.exists(FRAME_PATH):
            time.sleep(0.005)
            continue

        try:
            frame_rgb = np.load(FRAME_PATH)
            os.remove(FRAME_PATH)
        except Exception as e:
            continue

        # 3. SANITY CHECKS
        if frame_rgb.dtype != np.uint8:
            frame_rgb = frame_rgb.astype(np.uint8)

        if frame_rgb.ndim != 3 or frame_rgb.shape[2] != 3:
            continue

    
        timestamp_ms = int(time.perf_counter() * 1000)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        output = {
            "gesture": None,
            "score": None,
            "error": None
        }

        
        try:
            result = recognizer.recognize_for_video(mp_image, timestamp_ms)

            if result.gestures and len(result.gestures[0]) > 0:
                top = result.gestures[0][0]
                output["gesture"] = top.category_name
                output["score"] = float(top.score)

        except Exception as e:
            output["error"] = str(e)

        print(json.dumps(output))
        
    fps = (50.0/((int(time.perf_counter() * 1000) - start_time)))*1000
        
        
print("FPS approx: " + str(fps))
